# Project Phase: Market Demand Mapping (TomTom API)
## Objective
The primary goal of this notebook section is to identify High-Traffic Demand Hubs across Bangalore. We are specifically looking for locations where large numbers of potential customers gather during peak meal times:

Tech Parks: To capture the 1:00 PM - 2:00 PM office lunch crowd.

Metro Stations: To capture the "Grab-and-Go" morning and evening commuter crowd.

## Methodology
We use the TomTom POI (Point of Interest) Search API to programmatically fetch geographic data that isn't available in our static Zomato dataset.

Query Strategy: Searching for keywords "Tech Park" and "Metro Station" within a 20km radius of the city center.

Data Enrichment: Extracting postalCode and municipalitySubdivision (Neighborhood) to create a common link with the Zomato "Supply" dataset.

## Expected Outcome
By the end of this script, we will have two structured DataFrames (tech_df and metro_df). These will be used to calculate the Opportunity Score by comparing these "Demand Points" against the "Restaurant Supply" from Zomato.

In [15]:
import requests
import pandas as pd


## 1. TOMTOM API CONFIGURATION

In [16]:
API_KEY = "WxH16MZLVvQztZTNWnR7hPglcqjvet8c" 

## Searches for Points of Interest (POIs) in Bangalore using TomTom API.
## Used to identify high-traffic areas like Tech Parks and Metro Stations

### **📖 Data Dictionary: TomTom API Output**

| Column Name | Data Type | Description | Example Value |
| :--- | :--- | :--- | :--- |
| **Name** | String | The official name of the Point of Interest (POI). | "Manyata Tech Park" |
| **Address** | String | The full, formatted physical address. | "Nagawara, Bangalore 560045" |
| **Lat** | Float | The precise North/South geographic coordinate. | 12.9716 |
| **Lon** | Float | The precise East/West geographic coordinate. | 77.5946 |
| **Dist_Meters** | Float | Straight-line distance from the search center in meters. | 8450.5 |
| **postalCode** | String | The 6-digit Bangalore postal code (PIN code). | "560045" |
| **neighborhood** | String | The specific local area or sub-locality name. | "Nagawara" |

In [17]:


def search_bangalore_pois(query_text):
    # This URL searches for specific keywords near a location
    url = f"https://api.tomtom.com/search/2/poiSearch/{query_text}.json"
    
    params = {
        'key': API_KEY,
        'lat': 12.9716,      # Bangalore Latitude
        'lon': 77.5946,      # Bangalore Longitude
        'radius': 20000,     # 20km radius (covers almost all of Bangalore)
        'limit': 50,         # Start with 50 results to be safe
        'countrySet': 'IN'   # Limits search to India
    }
    
    try:
        response = requests.get(url, params=params)
        
        # Check if the key is working
        if response.status_code == 403:
            return "Error: Your API Key is invalid or not yet active. Wait 10 mins after creating it."
        
        data = response.json()
        results = data.get('results', [])
        
        if not results:
            return "No results found. Try broadening your query text."

        # Extracting into a clean list
        output = []
        for r in results:
            output.append({
                'Name': r['poi'].get('name'),
                'Address': r['address'].get('freeformAddress'),
                'Dist_Meters': r.get('dist'),
                'postalCode': r['address'].get('postalCode'),
                'neighborhood': r['address'].get('municipalitySubdivision'),
                'lat': r['position'].get('lat'),
                'lon': r['position'].get('lon')
            })
        return pd.DataFrame(output)

    except Exception as e:
        return f"Request failed: {e}"



# --- EXECUTE ANALYSIS ---

In [18]:

# Test 1: Tech Parks

#print("Searching for Tech Parks...")
tech_df = search_bangalore_pois("Tech Park")
#print(tech_df)

# Test 2: Metro Stations

#print("\nSearching for Metro Stations...")
metro_df = search_bangalore_pois("Metro Station")
#print(metro_df)

# TomTom API Output: Tech Park Sample

In [19]:
tech_df.head()

,Name,Address,Dist_Meters,postalCode,neighborhood,lat,lon
0,Global Tech Park,"Hosur Main Road, Pukhraj Layout, Bengaluru 560...",4089.415685,560030,None,12.936621,77.606255
1,Global Tech Park,"Langford Road, Langford Gardens, Shanti Nagar,...",1184.867470,560027,Shanti Nagar,12.962077,77.599505
2,Vikas Tech Park,"135, 1st Main Road, Industrial Layout, Koraman...",4740.531526,560095,Koramangala,12.934483,77.616121
3,Bagmane Tech Park,"Laxmi Sagar Layout, Mahadevapura, Bengaluru 56...",10808.003568,560048,Mahadevapura,12.983218,77.693631
4,Prestige Tech Park,"Outer Ring Road, Kodbisanhalli, Panatur, Benga...",11533.625344,560037,Panatur,12.942411,77.696733


# TomTom API Output: Metro Station Sample

In [20]:
metro_df.head()

,Name,Address,Dist_Meters,postalCode,neighborhood,lat,lon
0,Sir M Visveshwaraya Metro Station,"Post Office Road, Channarayapatna, Sampangiram...",1112.797724,560051,Sampangiram Nagar,12.974630,77.584812
1,Cubbon Park Metro Station,"Cubbon Road, Shivaji Nagar, Bengaluru 560051, ...",1130.797984,560051,Shivaji Nagar,12.979536,77.601126
2,MG Road Metro Station,"Mahatma Gandhi Road, Tasker Town, Shivaji Naga...",1386.753263,560051,Shivaji Nagar,12.975511,77.606752
3,Vidhana Soudha Metro Station,"Dr Ambedkar Road, High Court, Sampangiram Naga...",834.032500,560051,Sampangiram Nagar,12.978572,77.591762
4,Dr B R Ambedkar Station Vidhana Soudha,"Doctor B R Ambedakar Road, Vidhana Soudha, Sam...",968.888057,560001,Sampangiram Nagar,12.980114,77.592698


#  Specifically remove rows where 'neighborhood' is None/NaN

In [21]:

tech_df = tech_df.dropna(subset=['neighborhood'])

In [22]:

metro_df = metro_df.dropna(subset=['neighborhood'])

#  Standardize the keys

In [23]:

tech_df['neighborhood'] = tech_df['neighborhood'].str.lower().str.strip()

In [24]:
metro_df['neighborhood'] = metro_df['neighborhood'].str.lower().str.strip()

# Double-check the result

In [25]:

print(f"Rows remaining after cleaning: {len(tech_df)}")

Rows remaining after cleaning: 48


In [26]:

print(f"Rows remaining after cleaning: {len(metro_df)}")

Rows remaining after cleaning: 46
